# CEMS API ingestion

## Executive summary
Read one Copernicus EMS Rapid Mapping activation from the public dashboard API. Published vector layers (`vt` GeoJSON) are cached under `data/cache/cems/{activation_id}/`. This notebook does not replace `input/copernicus/{activation_id}/data.gpkg`. Notebooks 02 and 03 still read that local file.

## Prerequisites
Install `requirements.txt`. The activation ID comes from `config/params.yaml`. The API needs no token. COG orthophotos and product ZIP packages are listed and not downloaded.

## Parameters

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.copernicus_loader import load_params

params = load_params(PROJECT_ROOT)
ACTIVATION_ID = params['activation_id']
print({
    'activation_id': ACTIVATION_ID,
    'hazard_type': params.get('hazard_type'),
    'country_code': params.get('country_code'),
})

{'activation_id': 'EMSR884', 'hazard_type': 'earthquake', 'country_code': 'VEN'}


## Activation
One call to `public-activations/?code=` returns the event, its areas of interest, and every product.

In [3]:
from src.cems_api import activation_summary, fetch_activation

activation = fetch_activation(ACTIVATION_ID)
print(activation_summary(activation))

{'code': 'EMSR884', 'name': 'Earthquake in Venezuela', 'category': 'Earthquake', 'sub_category': 'Ground shaking', 'countries': 'Venezuela', 'event_time': '2026-06-24T16:04:00', 'closed': True, 'gdacs_id': 'EQ1548377', 'charter_number': '1036', 'n_aois': 13, 'n_products': 21}


## Areas of interest and products
`status_code` `F` is published. `N` is listed here and is not downloaded. `download_path` is the product ZIP. Vector GeoJSON links are in the layer index; COG rows usually have a storage name and no `url`.

In [4]:
from src.cems_api import layer_index, product_catalog

catalog = product_catalog(activation)
display(catalog)

layers = layer_index(activation)
display(layers)

,aoi_number,aoi_name,type,monitoring_number,status_code,delivery_time,n_vector_layers,n_cog_layers,download_path
0,0,Central Coastal Venezuela,GRM,0,F,2026-07-06T14:42:15.412962,2,0,https://rapidmapping.emergency.copernicus.eu/b...
1,1,Petare,GRA,0,N,2026-06-28T00:13:00,0,1,
2,2,Caracas,GRA,0,F,2026-06-26T04:01:10.948274,3,1,https://rapidmapping.emergency.copernicus.eu/b...
3,2,Caracas,GRA,1,F,2026-06-29T18:18:07.700477,4,1,https://rapidmapping.emergency.copernicus.eu/b...
4,3,Antimano,GRA,0,N,2026-06-26T02:32:00,0,1,
5,4,Maracay,GRA,0,N,2026-06-28T17:46:00,0,1,
6,5,Santa Cruz,GRA,0,F,2026-06-27T23:49:49.849118,2,1,https://rapidmapping.emergency.copernicus.eu/b...
7,5,Santa Cruz,GRA,1,N,2026-06-28T16:31:00,0,1,
8,6,Moron,GRA,1,F,2026-06-29T11:25:23.016205,3,1,https://rapidmapping.emergency.copernicus.eu/b...
9,6,Moron,GRA,0,F,2026-06-26T07:24:29.440918,3,1,https://rapidmapping.emergency.copernicus.eu/b...


,aoi_number,aoi_name,product_type,monitoring_number,status_code,format,layer,url
0,0,Central Coastal Venezuela,GRM,0,F,vt,EMSR884/AOI00/GRM_PRODUCT/EMSR884_AOI00_GRM_PR...,https://rapidmapping-viewer.s3.eu-west-1.amazo...
1,0,Central Coastal Venezuela,GRM,0,F,vt,EMSR884/AOI00/GRM_PRODUCT/EMSR884_AOI00_GRM_PR...,https://rapidmapping-viewer.s3.eu-west-1.amazo...
2,1,Petare,GRA,0,N,cog,EMSR884/AOI01/GRA_PRODUCT/EMSR884_AOI01_GRA_PR...,NaN
3,2,Caracas,GRA,0,F,cog,EMSR884/AOI02/GRA_PRODUCT/EMSR884_AOI02_GRA_PR...,NaN
4,2,Caracas,GRA,0,F,vt,EMSR884/AOI02/GRA_PRODUCT/EMSR884_AOI02_GRA_PR...,https://rapidmapping-viewer.s3.eu-west-1.amazo...
...,...,...,...,...,...,...,...,...
57,12,Caraballeda,GRA,0,F,cog,EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PR...,NaN
58,12,Caraballeda,GRA,0,F,vt,EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PR...,https://rapidmapping-viewer.s3.eu-west-1.amazo...
59,12,Caraballeda,GRA,0,F,vt,EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PR...,https://rapidmapping-viewer.s3.eu-west-1.amazo...
60,12,Caraballeda,GRA,0,F,vt,EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PR...,https://rapidmapping-viewer.s3.eu-west-1.amazo...


## Published vector layers
Only `vt` layers on products with `status_code` `F` are downloaded, into `data/cache/cems/{activation_id}/`. A file already in that folder is reused. One failed layer is printed and the rest continue.

In [5]:
from src.cems_api import load_published_vectors
from src.copernicus_loader import layer_summary

vector_layers, vector_errors = load_published_vectors(activation)
if vector_errors:
    print('Layer errors:')
    for message in vector_errors:
        print(message)
print({'published_vector_layers': len(vector_layers), 'failed_layers': len(vector_errors)})

if vector_layers:
    display(layer_summary(vector_layers))
    for name, layer in vector_layers.items():
        preview_cols = [column for column in layer.columns if column != 'geometry'][:8]
        print(name)
        display(layer[preview_cols].head())

Layer errors:
EMSR884/AOI00/GRM_PRODUCT/EMSR884_AOI00_GRM_PRODUCT_groundMovementA_v2_VT: HTTPSConnectionPool(host='rapidmapping-viewer.s3.eu-west-1.amazonaws.com', port=443): Read timed out. (read timeout=30)
{'published_vector_layers': 39, 'failed_layers': 1}


,layer,feature_count,geometry_type,crs,columns
0,EMSR884/AOI00/GRM_PRODUCT/EMSR884_AOI00_GRM_PR...,245,Polygon,EPSG:4326,"obj_type, or_src_id, cems_layer, aoi_number, a..."
1,EMSR884/AOI02/GRA_PRODUCT/EMSR884_AOI02_GRA_PR...,17,Polygon,EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."
2,EMSR884/AOI02/GRA_PRODUCT/EMSR884_AOI02_GRA_PR...,24,"MultiPolygon, Polygon",EPSG:4326,"obj_type, or_src_id, cems_layer, aoi_number, a..."
3,EMSR884/AOI02/GRA_PRODUCT/EMSR884_AOI02_GRA_PR...,9135,"LineString, MultiLineString",EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."
4,EMSR884/AOI02/GRA_MONIT01/EMSR884_AOI02_GRA_MO...,2,Point,EPSG:4326,"obj_type, obj_desc, elem_count, det_method, no..."
5,EMSR884/AOI02/GRA_MONIT01/EMSR884_AOI02_GRA_MO...,20,Point,EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."
6,EMSR884/AOI02/GRA_MONIT01/EMSR884_AOI02_GRA_MO...,22,Polygon,EPSG:4326,"obj_type, or_src_id, cems_layer, aoi_number, a..."
7,EMSR884/AOI02/GRA_MONIT01/EMSR884_AOI02_GRA_MO...,9683,"LineString, MultiLineString",EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."
8,EMSR884/AOI05/GRA_PRODUCT/EMSR884_AOI05_GRA_PR...,3,Point,EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."
9,EMSR884/AOI05/GRA_PRODUCT/EMSR884_AOI05_GRA_PR...,7347,"LineString, MultiLineString",EPSG:4326,"obj_type, name, info, simplified, damage_gra, ..."


EMSR884/AOI00/GRM_PRODUCT/EMSR884_AOI00_GRM_PRODUCT_notAnalysedA_v2_VT


,obj_type,or_src_id,cems_layer,aoi_number,aoi_name,product_type,monitoring_number,delivery_time
0,Not Analysed,997,EMSR884/AOI00/GRM_PRODUCT/EMSR884_AOI00_GRM_PR...,0,Central Coastal Venezuela,GRM,0,2026-07-06T14:42:15.412962
1,Not Analysed,997,EMSR884/AOI00/GRM_PRODUCT/EMSR884_AOI00_GRM_PR...,0,Central Coastal Venezuela,GRM,0,2026-07-06T14:42:15.412962
2,Not Analysed,997,EMSR884/AOI00/GRM_PRODUCT/EMSR884_AOI00_GRM_PR...,0,Central Coastal Venezuela,GRM,0,2026-07-06T14:42:15.412962
3,Not Analysed,997,EMSR884/AOI00/GRM_PRODUCT/EMSR884_AOI00_GRM_PR...,0,Central Coastal Venezuela,GRM,0,2026-07-06T14:42:15.412962
4,Not Analysed,997,EMSR884/AOI00/GRM_PRODUCT/EMSR884_AOI00_GRM_PR...,0,Central Coastal Venezuela,GRM,0,2026-07-06T14:42:15.412962


EMSR884/AOI02/GRA_PRODUCT/EMSR884_AOI02_GRA_PRODUCT_builtUpA_v1_VT


...

,obj_type,or_src_id,cems_layer,aoi_number,aoi_name,product_type,monitoring_number,delivery_time
0,Not Analysed,3,EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PR...,12,Caraballeda,GRA,0,2026-06-27T19:54:43.077956
1,Not Analysed,3,EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PR...,12,Caraballeda,GRA,0,2026-06-27T19:54:43.077956
2,Not Analysed,3,EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PR...,12,Caraballeda,GRA,0,2026-06-27T19:54:43.077956
3,Not Analysed,3,EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PR...,12,Caraballeda,GRA,0,2026-06-27T19:54:43.077956
4,Not Analysed,3,EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PR...,12,Caraballeda,GRA,0,2026-06-27T19:54:43.077956


EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PRODUCT_transportationA_v2_VT


,obj_type,name,info,simplified,damage_gra,det_method,notation,or_src_id
0,213-Airfield,Aeropuerto Internacional de Maiquetía Simón Bo...,2130-Airfield runways,Airfield,Damaged,Photo-interpretation,Not Applicable,994


EMSR884/AOI12/GRA_PRODUCT/EMSR884_AOI12_GRA_PRODUCT_transportationL_v2_VT


,obj_type,name,info,simplified,damage_gra,det_method,notation,or_src_id
0,"211-Highways, Streets and Roads",Unknown,21124-Cart Track,Tracks,No visible damage,Photo-interpretation,Not Applicable,994
1,"211-Highways, Streets and Roads",Unknown,21122-Local Road,Local roads,No visible damage,Photo-interpretation,Not Applicable,994
2,"211-Highways, Streets and Roads",Unknown,21120-Primary Road,Main roads,No visible damage,Photo-interpretation,Not Applicable,994
3,"211-Highways, Streets and Roads",Unknown,21122-Local Road,Local roads,No visible damage,Photo-interpretation,Not Applicable,994
4,"211-Highways, Streets and Roads",Unknown,21122-Local Road,Local roads,No visible damage,Photo-interpretation,Not Applicable,994


## Apply the damage filter used downstream
Same rule as the local notebook: keep rows where `damage_column` equals `damage_positive_value`. Official CEMS layers usually have no `damage` column (that column belongs to the local Overture file). When it is missing, every geometry is kept and the attribute names are printed.

In [6]:
from src.copernicus_loader import bbox_from_geodataframe, concat_layers, filter_impact

if not vector_layers:
    print('No published vector layers were loaded, so the damage filter was not applied.')
else:
    impact_all = concat_layers(vector_layers)
    damage_column = (params.get('impact') or {}).get('damage_column', 'damage')
    impact = filter_impact(impact_all, params)
    buffer_deg = float((params.get('impact') or {}).get('bbox_buffer_deg', 0.05))
    print({
        'features_loaded': len(impact_all),
        'features_after_damage_filter': len(impact),
        'damage_column': damage_column,
        'damage_column_present': damage_column in impact_all.columns,
        'columns': [column for column in impact_all.columns if column != 'geometry'],
        'bounds_wgs84': impact.to_crs('EPSG:4326').total_bounds.tolist() if not impact.empty else None,
        'bbox_with_buffer': bbox_from_geodataframe(impact, buffer_deg) if not impact.empty else None,
    })

{'features_loaded': 53093, 'features_after_damage_filter': 53091, 'damage_column': 'damage', 'damage_column_present': False, 'columns': ['obj_type', 'or_src_id', 'cems_layer', 'aoi_number', 'aoi_name', 'product_type', 'monitoring_number', 'delivery_time', 'copernicus_layer', 'name', 'info', 'simplified', 'damage_gra', 'det_method', 'notation', 'dmg_src_id', 'cd_value', 'real', 'obj_desc', 'elem_count'], 'bounds_wgs84': [-68.8580017, 9.81542903, -66.25415974, 10.84657494], 'bbox_with_buffer': [-68.9080017, 9.76542903, -66.20415974000001, 10.89657494]}


## Copernicus statistics
Figures already published on the activation and on each published GRA or GRM product. These are Copernicus totals, not the administrative overlay from notebook 02.

In [7]:
from src.cems_api import impact_stats_table

stats = impact_stats_table(activation)
display(stats)

,scope,aoi_number,aoi_name,product_type,monitoring_number,thematic,subclass,unit,total,affected
0,activation,NaN,NaN,NaN,NaN,Roads [km],NaN,NaN,-,NaN
1,activation,NaN,NaN,NaN,NaN,max_extent,NaN,NaN,-,NaN
2,activation,NaN,NaN,NaN,NaN,Population [No.],NaN,NaN,-,NaN
3,activation,NaN,NaN,NaN,NaN,Built-up area [ha],NaN,NaN,-,NaN
4,activation,NaN,NaN,NaN,NaN,Identified buildings [No.],NaN,NaN,3119,NaN
...,...,...,...,...,...,...,...,...,...,...
58,product,12.0,Caraballeda,GRA,1.0,Blocked road / interruption,None,NaN,NA,53.0
59,product,12.0,Caraballeda,GRA,0.0,NA,None,NaN,NA,NaN
60,product,12.0,Caraballeda,GRA,0.0,Built-up,Residential Buildings,ha,550.9,550.9
61,product,12.0,Caraballeda,GRA,0.0,Transportation,Airfield runways,ha,601.3,601.3
